# 00 — Environment

**Purpose.** Bootstrap the Colab runtime for the NRC-Cal project: detect the
runtime, mount Google Drive, resolve/scaffold the project directory, install
dependencies, verify PyTorch/CUDA **and that the attached GPU is a T4**,
detect system resources, and save a reproducible environment report.

**No equations in this notebook** — pure infrastructure. NRC math starts in
`05_compute_NRC.ipynb`, gated on the exact NRC1/NRC2/NRC3 formulas being
confirmed from source.

**Expected runtime:** under 2 minutes on a fresh Colab runtime.
**GPU:** targets an NVIDIA T4 (Colab "T4 GPU" runtime). Not required for this
notebook itself (nothing is loaded onto the GPU here), but verified so later
notebooks can trust the report this one saves.

Every code cell is written to be **re-runnable independently and out of
order** — each re-establishes `PROJECT_ROOT` / `env_utils` if missing from
the namespace.


## Reading files from your MacBook while compute runs on Colab's cloud T4

This needs to be said precisely, because "Colab GPU + my local files" has
exactly one practical answer, not three equally-good ones:

**The T4 GPU runs in Google's cloud. It cannot reach into your MacBook's
filesystem directly** — there is no direct mount, no matter which option
below you pick. What actually solves "I want to keep editing files locally
and have Colab see them" is:

> **Install [Google Drive for Desktop](https://www.google.com/drive/download/)
> on your Mac, in "Mirror files" or "Stream files" mode.** This creates a
> folder on your Mac (e.g. `~/Google Drive/My Drive/project`) that is
> transparently, automatically synced with Drive in the cloud. You edit
> files in that folder **exactly as if it were a normal local folder** —
> because it is one — and Drive for Desktop pushes changes to the cloud in
> the background. Colab (Option A, below) then mounts that same Drive and
> sees the identical files, usually within seconds of you saving.

This is **Option A**, and it is the one to use for this project. Options B
(GitHub) and C (SSH/rsync) are documented and implemented below for
completeness / cases where you don't want a Drive dependency, but neither
gives you the "just save the file locally, Colab sees it" workflow that
Drive for Desktop does — B requires an explicit `git push`/`pull` each time,
and C requires your Mac to be reachable from Google's cloud (a tunnel like
Tailscale — genuinely not solvable in software running only on the Colab
side, since Colab has no route to a NAT'd home Mac).

**Practical setup, once:**
1. Install Drive for Desktop, sign in, enable Mirror or Stream for `My Drive`.
2. Move (or symlink) this `project/` folder into `~/Google Drive/My Drive/project`.
3. Keep editing locally on your Mac as normal.
4. In Colab, run Step 2 below — it mounts the same Drive and finds the same
   folder automatically.


## Step 1 — Detect runtime and make `src/` importable

In [ ]:
import sys
from pathlib import Path

def _find_repo_root(start: Path) -> Path:
    '''Walk upward from `start` until a folder containing `src/utils/env_utils.py` is found.'''
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "utils" / "env_utils.py").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the project repo (looked for src/utils/env_utils.py "
        f"above {start}). If this is a fresh Colab runtime, run Step 2/3 first "
        "to mount Drive, then re-run this cell."
    )

_here = Path.cwd()
try:
    REPO_ROOT = _find_repo_root(_here)
except FileNotFoundError as exc:
    print(f"[info] {exc}")
    REPO_ROOT = _here

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

try:
    from src.utils import env_utils
    print(f"[ok] src.utils.env_utils imported from {REPO_ROOT}")
except ModuleNotFoundError as exc:
    env_utils = None
    print(f"[info] env_utils not importable yet ({exc}). Expected until Drive is mounted (Step 2).")

print(f"In Colab: {env_utils.is_colab() if env_utils else 'unknown (env_utils not loaded)'}")


## Step 2 — Mount Google Drive (Option A — the one to use, see above)

If you've set up Drive for Desktop on your Mac and placed `project/` under
`My Drive`, this cell finds the exact same files your Mac is editing.

In [ ]:
if "env_utils" not in dir() or env_utils is None:
    import sys as _sys; from pathlib import Path as _Path
    if str(_Path.cwd()) not in _sys.path: _sys.path.insert(0, str(_Path.cwd()))
    from src.utils import env_utils

DRIVE_ROOT = env_utils.mount_google_drive()
print(f"Drive root: {DRIVE_ROOT}" if DRIVE_ROOT else "Drive not mounted (not in Colab, or unused).")


## Step 3 — Resolve project root & scaffold directories

In [ ]:
if "env_utils" not in dir() or env_utils is None:
    from src.utils import env_utils

LOCAL_FALLBACK = REPO_ROOT if "REPO_ROOT" in dir() else Path.cwd()

PROJECT_ROOT = env_utils.locate_project_root(
    drive_root=DRIVE_ROOT if "DRIVE_ROOT" in dir() else None,
    repo_dir_name="project",
    local_fallback=LOCAL_FALLBACK,
)
PATHS = env_utils.ensure_project_structure(PROJECT_ROOT)

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
for name, p in PATHS.items():
    print(f"  {name:12s} -> {p}")


## Step 4 — Option B: GitHub sync (optional, off by default)

In [ ]:
import yaml

config_path = PROJECT_ROOT / "configs" / "config.yaml"
if config_path.exists():
    cfg = yaml.safe_load(config_path.read_text())
    gh_cfg = cfg.get("sync", {}).get("github", {})
else:
    print(f"[warn] {config_path} not found -- skipping Option B.")
    gh_cfg = {"enabled": False}

if gh_cfg.get("enabled") and gh_cfg.get("repo_url"):
    ok = env_utils.clone_or_pull_repo(
        repo_url=gh_cfg["repo_url"], dest=PROJECT_ROOT, branch=gh_cfg.get("branch", "main"),
    )
    print(f"GitHub sync {'succeeded' if ok else 'failed -- see logged error above'}.")
else:
    print("Option B (GitHub) disabled or repo_url not set -- skipping.")


## Step 5 — Option C: SSH / rsync (optional, off by default, needs a reachable Mac)

In [ ]:
ssh_cfg = cfg.get("sync", {}).get("ssh_rsync", {}) if "cfg" in dir() else {"enabled": False}

if ssh_cfg.get("enabled") and ssh_cfg.get("remote_host"):
    rsync_cmd = env_utils.build_rsync_command(
        remote_user=ssh_cfg["remote_user"], remote_host=ssh_cfg["remote_host"],
        remote_path=ssh_cfg["remote_path"], local_path=PROJECT_ROOT,
        ssh_port=ssh_cfg.get("ssh_port", 22),
    )
    print(f"Running: {rsync_cmd}")
    result = env_utils.run_rsync(rsync_cmd, timeout_s=15)
    print("Success" if result["success"] else f"Failed/timed out: {result['stderr']}")
else:
    print("Option C (SSH/rsync) disabled -- skipping. Requires Tailscale/tunnel; see markdown above.")


## Step 6 — Install project dependencies

Uses the real dependency versions verified against the actual QRT/PCS repos
(`pytorch_lightning`, `pyro-ppl`, `openml==0.13.1`, etc.) — not generic
guesses. See `requirements.txt` for the full, commented list.

In [ ]:
req_path = PROJECT_ROOT / "requirements.txt"
if req_path.exists():
    print(f"Installing from {req_path} ...")
    %pip install -q -r "{req_path}"
    print("Dependency install complete.")
else:
    print(f"[warn] {req_path} not found -- skipping install.")


## Step 7 — Verify PyTorch / CUDA / T4

In [ ]:
if "env_utils" not in dir() or env_utils is None:
    from src.utils import env_utils

pytorch_info = env_utils.verify_pytorch_cuda()
for k, v in pytorch_info.items():
    print(f"{k:16s}: {v}")

gpu_check = env_utils.verify_gpu_matches_target(target_substring="T4")
print()
for k, v in gpu_check.items():
    print(f"{k:16s}: {v}")
if gpu_check["has_gpu"] and not gpu_check["matches_target"]:
    print("\n[warn] Attached GPU is not a T4. In Colab: Runtime > Change runtime type "
          "> Hardware accelerator > T4 GPU, then Runtime > Restart and re-run.")
elif not gpu_check["has_gpu"]:
    print("\n[warn] No GPU attached at all. In Colab: Runtime > Change runtime type > T4 GPU.")


## Step 8 — Enable mixed precision defaults (T4-aware)

T4 is a Turing-generation GPU (compute capability 7.5): it has fp16 tensor
cores but **no native bf16 tensor core support** (that arrived with Ampere,
capability 8.0+). Later notebooks that use autocast on this project should
therefore use `torch.float16`, never `torch.bfloat16`, when running on this
runtime.

In [ ]:
mp_info = env_utils.enable_mixed_precision_defaults()
for k, v in mp_info.items():
    print(f"{k:24s}: {v}")
print("\nRecommended autocast dtype on T4: torch.float16 (bf16 is not natively accelerated on Turing).")


## Step 9 — Detect system resources (RAM / disk)

In [ ]:
sys_info = env_utils.detect_system_resources()
for k, v in sys_info.items():
    print(f"{k:18s}: {v}")


## Step 10 — Assemble & save the full environment report

In [ ]:
if "env_utils" not in dir() or env_utils is None:
    from src.utils import env_utils

report = env_utils.build_environment_report(project_root=PROJECT_ROOT)
report_path = env_utils.save_environment_report(report, PATHS["logs"] / "environment_report.json")
env_utils.pretty_print_report(report)
print(f"\nSaved to: {report_path}")


## Next steps

**Next:** `01_download_repositories.ipynb` — clones the two real Vekteur
repos, installs their exact dependency stack, and verifies structure.

**Important finding already baked into notebook 01:** neither repo ships
pretrained checkpoints. Notebook 03 will train small pilot models (the
`uci` dataset group — 12 small datasets, no OpenML account needed) rather
than downloading them.
